# task21: Can both teams’ recent records predict Team A goals minus Team B goals?

The analysis uses eight pre-match predictors. Targets include extra time and exclude shootout kicks.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'analysis.py').exists(): ROOT=ROOT/'Assignment3'
assert (ROOT/'analysis.py').exists()
sys.path.insert(0,str(ROOT.resolve()))
from analysis import prepare_data, run_task, TASK1_X, TASK2_X

one,two=prepare_data()
df=one
features=TASK1_X
print(features)
print(df.groupby('split').size().to_string())

['a_pre_gf', 'a_pre_ga', 'a_pre_win_rate', 'a_pre_clean_sheet_rate', 'b_pre_gf', 'b_pre_ga', 'b_pre_win_rate', 'b_pre_clean_sheet_rate']
split
test          22
train         56
validation    26


## T1–T3: baseline, multiple regression and exploration

Fit ordinary least squares (OLS) only on training rows. Compare classroom transformations and a training-mean baseline using identical splits. All primary regressions retain eight X variables. MAE is average absolute error; MSE averages squared errors; RMSE is its square root in goals. NRMSE divides RMSE by the training target range (a training-only adaptation of the classroom denominator). R² compares errors with predicting the evaluation-set mean; it can be negative. No predictor imputation is needed.

In [2]:
summary=run_task(df,features,'goal_difference','task21')
out=ROOT/'results/task21'
print(pd.read_csv(out/'training_descriptive_statistics.csv').to_string(index=False))

Unnamed: 0  a_pre_gf  a_pre_ga  a_pre_win_rate  a_pre_clean_sheet_rate  b_pre_gf  b_pre_ga  b_pre_win_rate  b_pre_clean_sheet_rate  goal_difference
     count 56.000000 56.000000       56.000000               56.000000 56.000000 56.000000       56.000000               56.000000        56.000000
      mean  1.821429  1.125000        0.485714                0.314286  1.607143  1.160714        0.442857                0.367857         0.857143
       std  0.805146  0.658994        0.269294                0.237738  0.728599  0.675191        0.268908                0.234410         2.152618
       min  0.200000  0.200000        0.000000                0.000000  0.200000  0.000000        0.000000                0.000000        -4.000000
       25%  1.400000  0.600000        0.400000                0.150000  1.150000  0.600000        0.200000                0.200000         0.000000
       50%  1.800000  1.000000        0.400000                0.400000  1.600000  1.000000        0.400000      

![Training scatterplots](results/task21/training_scatterplots.png)

Relationships are not causal effects.

## Simple versus multiple linear regression

This supplementary comparison uses one raw predictor versus all eight raw predictors. The single predictor is previous-five-match mean goals scored: `team_pre_gf` for Task 2.2 (the meaning of `team_previous_mean_goals_for`), or `a_pre_gf` for Team A in Task 2.1. It is fixed by the research question, not chosen by test performance. Both models use OLS with an intercept, the same training rows and exactly the same test rows. No transformations are used in this comparison.

MAE and RMSE are in goals; NRMSE divides RMSE by the same training-response range. Lower values indicate smaller prediction errors. Both models belong to the same algorithm family. This comparison does not establish compliance with a requirement for different algorithms, and the one-predictor model does not replace the required eight-predictor analysis. Evaluation is retrospective because these test outcomes have previously been inspected.

In [3]:
comparison=pd.read_csv(out/'simple_vs_multiple_metrics.csv')
print(comparison[['model','predictor_count','MAE','RMSE','NRMSE']].to_string(index=False))
print((out/'simple_vs_multiple_details.json').read_text())
simple_rmse,multiple_rmse=comparison.RMSE
print(f'RMSE reduction from adding seven predictors: {simple_rmse-multiple_rmse:.6f} goals')
print('The eight-predictor model has lower test RMSE.' if multiple_rmse < simple_rmse else 'The eight-predictor model does not have lower test RMSE.')
print('This describes this reused test period; it does not guarantee future performance.')

           model  predictor_count      MAE     RMSE    NRMSE
  simple_raw_ols                1 1.493509 1.965688 0.196569
multiple_raw_ols                8 1.373744 1.757800 0.175780
{
  "simple_feature": "a_pre_gf",
  "feature_meaning": "Mean goals scored in the previous five completed internationals",
  "simple_intercept": -0.35724016347463705,
  "simple_coefficient": 0.6667200897507811,
  "multiple_features": [
    "a_pre_gf",
    "a_pre_ga",
    "a_pre_win_rate",
    "a_pre_clean_sheet_rate",
    "b_pre_gf",
    "b_pre_ga",
    "b_pre_win_rate",
    "b_pre_clean_sheet_rate"
  ],
  "multiple_intercept": 2.5712292069734377,
  "multiple_coefficients": [
    0.23543675768757832,
    -0.25640406545192906,
    2.2039527051808925,
    -1.2719398879854564,
    -0.18312990338945778,
    -0.26190939790180934,
    -2.9689003602347928,
    -1.6639208350028405
  ],
  "training_rows": 56,
  "test_rows": 22,
  "nrmse_denominator": 10.0,
  "note": "Same training and test rows; raw predictors and r

## T4–T6: log transformation

Both tasks now use nonnegative team records. Use log(1 + X) to handle zeros; this is a stated adaptation of the classroom log transform. Leave the response untransformed, including signed goal differences. Predictor skewness is not automatically a regression-assumption violation.

## T7–T8: correlation and reduction

![Correlations](results/task21/training_correlation.png)

The teaching exercise screens training correlations above |r| = 0.8 and retains the earlier-listed column. This is not a full diagnosis. Reduced models do not replace the required eight-predictor primary models.

In [4]:
print((out/'t8_reduction_exercise.json').read_text())

{
  "retained_features": [
    "a_pre_gf",
    "a_pre_ga",
    "a_pre_clean_sheet_rate",
    "b_pre_gf",
    "b_pre_ga",
    "b_pre_win_rate",
    "b_pre_clean_sheet_rate"
  ],
  "validation": {
    "MAE": 1.1280250488541819,
    "MSE": 2.3778602285939097,
    "RMSE": 1.5420312022115212,
    "NRMSE": 0.15420312022115212,
    "R2": 0.3180171766951706
  },
  "note": "Classroom exercise only. Not eligible for final selection because the brief requires eight predictors."
}


## T9–T10: standardisation and Yeo–Johnson

Fit transformation parameters on training X only. Scaling OLS should not change fitted predictions apart from numerical precision. Log1p and Yeo–Johnson are transformations, not different regression algorithms. At the group’s request, no Ridge or other advanced algorithm is used. Full satisfaction of the HD competing-algorithms criterion is not claimed.

In [5]:
print(pd.read_csv(out/'validation_comparison.csv').to_string(index=False))
print('Best regression:',summary['selected_model'])
print('Validation recommendation including baseline:',summary['validation_recommendation'])

                 model      MAE      MSE     RMSE    NRMSE        R2
               raw_ols 1.190680 2.638042 1.624205 0.162420  0.243396
      standardised_ols 1.190680 2.638042 1.624205 0.162420  0.243396
        yeojohnson_ols 1.065532 2.357881 1.535539 0.153554  0.323747
             log1p_ols 1.049218 2.275830 1.508585 0.150859  0.347280
training_mean_baseline 1.543956 4.036892 2.009202 0.200920 -0.157802
Best regression: log1p_ols
Validation recommendation including baseline: log1p_ols


![Validation comparison](results/task21/validation_comparison.png)

## Equation and training summary

Coefficients multiply the selected transformed X values. Statsmodels verifies the same OLS model, not a competing algorithm. Training adjusted R² differs from evaluation R². Usual standard errors/p-values may be unreliable with repeated teams and paired rows; no strong independent-sample significance claims are made.

In [6]:
print((out/'selected_equation.txt').read_text())
print((out/'statsmodels_training_summary.txt').read_text())

Predicted goal_difference = 2.791795 +1.162451 * a_pre_gf -1.340110 * a_pre_ga +1.724601 * a_pre_win_rate -2.117131 * a_pre_clean_sheet_rate -0.767647 * b_pre_gf +0.137228 * b_pre_ga -3.533180 * b_pre_win_rate -1.370467 * b_pre_clean_sheet_rate
Features in this equation are log(1 + raw historical feature).
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.303
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     2.550
Date:                Tue, 13 Oct 2026   Prob (F-statistic):             0.0214
Time:                        15:15:32   Log-Likelihood:                -111.80
No. Observations:                  56   AIC:                             241.6
Df Residuals:                      47   BIC:                             259.8
Df Model:                           8                       

## Later-period evaluation (reused)

This run selects on validation RMSE. Nevertheless, the project previously inspected the evaluation outcomes: these are exploratory retrospective results. Predictions are not rounded or clipped to improve metrics. Negative goal differences are legitimate; negative predicted goal counts are an OLS limitation.

In [7]:
table=pd.read_csv(out/'test_metrics.csv')
print(table.to_string(index=False))
print('RMSE improvement:',table.RMSE.iloc[1]-table.RMSE.iloc[0])
print('Negative predictions:',summary['negative_test_predictions'])
print(pd.read_csv(out/'test_predictions.csv').to_string(index=False))

                 model      MAE      MSE     RMSE    NRMSE        R2
             log1p_ols 1.332761 2.799559 1.673188 0.167319 -0.044708
training_mean_baseline 1.487013 3.494434 1.869341 0.186934 -0.304014
RMSE improvement: 0.1961523927780795
Negative predictions: 4
match_id       date      team_a      team_b  actual  predicted  baseline  residual
    M083 2026-07-02    Portugal     Croatia       1   1.085832  0.857143 -0.085832
    M084 2026-07-02       Spain     Austria       3   0.606247  0.857143  2.393753
    M085 2026-07-02 Switzerland     Algeria       2   1.251109  0.857143  0.748891
    M086 2026-07-03   Argentina  Cape Verde       1   2.262194  0.857143 -1.262194
    M087 2026-07-03   Australia       Egypt       0   0.136913  0.857143 -0.136913
    M088 2026-07-03    Colombia       Ghana       1   2.185428  0.857143 -1.185428
    M089 2026-07-04      Canada     Morocco      -3   1.072044  0.857143 -4.072044
    M090 2026-07-04    Paraguay      France      -1  -1.481468  0.85

![Diagnostics](results/task21/test_diagnostics.png)

![Error story](results/task21/match_error_story.png)

Task 2.2 averages paired team errors within each match. Positive bars mean regression helped relative to the baseline; this is not a significance test.

## Discussion prompts

- Is any baseline improvement practically meaningful?
- How can RMSE improve over a training-mean baseline while R² remains negative? The comparison constants differ.
- Which matches dominate errors? Map IDs through `matches_104.csv`.
- Discuss friendlies, different opponent strength, five-game variability, extra time, omitted injuries/line-ups, reused evaluation data and post-event source corrections.
- A/B order can affect the signed goal-difference analysis; paired rows and repeated teams affect independence.
- Weak prediction is a valid result, not evidence to conceal.

In [8]:
import inspect
print(inspect.getsource(run_task))

def run_task(df, features, target, task):
    out = RESULTS / task
    out.mkdir(parents=True, exist_ok=True)
    train, valid, test = [df[df['split'] == s].copy() for s in ['train', 'validation', 'test']]
    assert len(features) == 8
    assert not set(train.match_id) & set(test.match_id)
    assert not set(valid.match_id) & set(test.match_id)
    assert pd.to_datetime(train.date).max() < pd.to_datetime(valid.date).min()
    assert pd.to_datetime(valid.date).max() < pd.to_datetime(test.date).min()
    assert np.isfinite(df[features]).all().all(), 'Do not impute missing historical data'
    xt, xv, xe = [part[features].to_numpy() for part in [train, valid, test]]
    yt, yv, ye = [part[target].to_numpy() for part in [train, valid, test]]
    target_range = float(np.ptp(yt))
    assert target_range > 0
    pd.DataFrame({'feature':features, 'missing_cells':0, 'imputation_used':False}).to_csv(out/'missingness.csv', index=False)
    train[features+[target]].describe().to_csv(out/'training